# Citation growth
Section 4.4.2 and Appendix A.4 (Table 5).

Default paths are relative: inputs under `data/`, results under
`outputs/07_citation_growth/`. No data are included.
Required files under each `pred/` and `norm/` directory:
- `dataset/csv/papers.csv`: `id`, `publication_date`, `total_citations`.
- `dataset/csv/papers_collect_after_6months.csv`: `old_id`, `total_citations`.
- `metadata/{pred,norm}_crossref_meta.csv`: `id`, `publication_date`, `source`.

Keys are local to each dataset. Citation snapshots are paired by `id`/`old_id`.
Only aggregate tables are displayed and saved to the configured results directory.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

# Default paths are relative to the repository root.
REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
OUTPUT_DIR = REPO_ROOT / 'outputs' / '07_citation_growth'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = {"Predatory": "pred", "Normal": "norm"}
FIRST_COLLECTION = {"Predatory": pd.Timestamp("2024-09-09"),
                    "Normal": pd.Timestamp("2024-12-09")}
SECOND_COLLECTION = pd.Timestamp("2025-07-07")
OBSERVATION_DAYS = {label: (SECOND_COLLECTION-first).days
                    for label, first in FIRST_COLLECTION.items()}
DAYS_PER_YEAR = 365.25
DAYS_PER_MONTH = 30.4375

def save_table(frame, name):
    # Only aggregate statistics and anonymous case metrics are exported.
    frame.to_csv(OUTPUT_DIR/(name+".csv"), index=False)
    display(frame)

def monthly_change(value, label):
    return value*DAYS_PER_MONTH/OBSERVATION_DAYS[label]


In [ ]:
def match_papers(baseline, followup):
    for frame, key in [(baseline, "id"), (followup, "old_id")]:
        if frame[key].isna().any() or frame[key].duplicated().any():
            raise ValueError(f"Matching key {key!r} must be nonmissing and unique.")
    return baseline[["id", "publication_date", "total_citations"]].merge(
        followup[["old_id", "total_citations"]], left_on="id", right_on="old_id",
        suffixes=("_t1", "_t2"), validate="one_to_one"
    )

def finite_counts(frame):
    return np.isfinite(frame[["total_citations_t1", "total_citations_t2"]]).all(axis=1)


paper_data = {}
for label, group in DATASETS.items():
    folder = DATA_DIR/group/"dataset"/"csv"
    baseline = pd.read_csv(folder/"papers.csv", usecols=["id", "publication_date", "total_citations"])
    followup = pd.read_csv(folder/"papers_collect_after_6months.csv", usecols=["old_id", "total_citations"])
    for frame in (baseline, followup):
        frame["total_citations"] = pd.to_numeric(frame["total_citations"], errors="coerce")
    paper_data[label] = {"baseline": baseline, "followup": followup,
                         "matched": match_papers(baseline, followup)}


## Papers labelled 2024
The first four-digit year matching 19xx or 20xx in the original publication-date
field identifies the cohort. Require finite counts in both snapshots; retain
declines and zero gains. Welch's test compares raw gains. Paper-level Cohen's d
uses the square root of the unweighted mean of the two sample variances.
The trimmed mean removes 10% from each tail. A percentile bootstrap resamples
each set independently 5,000 times with a reproducible random generator.

In [ ]:
BOOTSTRAP_RESAMPLES = 5000
BOOTSTRAP_SEED = 0
gain_2024 = {}
rows = []
for label, data in paper_data.items():
    paired = data["matched"]
    years = pd.to_numeric(paired["publication_date"].astype(str).str.extract(
        r"((?:19|20)\d{2})", expand=False), errors="coerce")
    selected = paired.loc[years.eq(2024) & finite_counts(paired)].copy()
    selected["gain"] = selected.total_citations_t2-selected.total_citations_t1
    gain_2024[label] = selected
    rows.append({"Dataset": label, "N": len(selected), "Window days": OBSERVATION_DAYS[label],
                 "Mean gain": selected.gain.mean(), "Monthly mean gain": monthly_change(selected.gain.mean(), label),
                 "Trimmed mean (10%)": stats.trim_mean(selected.gain, 0.1)})
save_table(pd.DataFrame(rows), "paper_2024_summary")
p, n = [gain_2024[label].gain.to_numpy() for label in DATASETS]
t, pv = stats.ttest_ind(p, n, equal_var=False)
d = (p.mean()-n.mean())/np.sqrt((p.var(ddof=1)+n.var(ddof=1))/2)
rng = np.random.default_rng(BOOTSTRAP_SEED)
differences = np.array([rng.choice(p, len(p), replace=True).mean()-
                        rng.choice(n, len(n), replace=True).mean()
                        for _ in range(BOOTSTRAP_RESAMPLES)])
lower, upper = np.percentile(differences, [2.5, 97.5])
save_table(pd.DataFrame([{"Welch t": t, "Welch p": pv, "Cohen d": d,
                          "Mean difference": p.mean()-n.mean(),
                          "Bootstrap CI lower": lower, "Bootstrap CI upper": upper}]), "paper_2024_tests")


## Top tenth by absolute citation gain
Within each 2024 cohort, rank by the raw citation increment, descending, and
select `ceil(0.10*N)` papers. Resolve ties by the dataset-local paper key.
Annualization multiplies the mean increment by 365.25 / observation-window days.

In [ ]:
top_rows = []
for label, frame in gain_2024.items():
    size = int(np.ceil(0.10*len(frame)))
    selected = frame.sort_values(["gain", "id"], ascending=[False, True], kind="stable").head(size)
    top_rows.append({"Dataset": label, "N selected": len(selected),
                     "Annualized mean gain": selected.gain.mean()*DAYS_PER_YEAR/OBSERVATION_DAYS[label]})
save_table(pd.DataFrame(top_rows), "paper_top_decile")


## Age-grouped monthly relative growth (Table 5)
Use the first stored metadata row per paper. Parse its date as YYYY-MM-DD,
keeping dates within 20 x 365 days before the first snapshot. Age uses a
365-day year and right-closed bins; exact age zero is outside the bins.
Exclude negative baseline counts and declines, but retain zero growth.
Relative gain divides by `max(baseline citations, 1)` and is converted to a
monthly rate using the set's observation window.

The full sample uses all stored date sources; the Crossref-only sample requires
the stored source tag `crossref`. Thus the full column is not exclusively
Crossref-dated. Mann-Whitney tests are two-sided and asymptotic. Apply Holm
correction across the five age bins separately within each sample.

In [ ]:
AGE_DAYS_PER_YEAR = 365.0
MAX_PUBLICATION_AGE_YEARS = 20
AGE_EDGES = [0, 0.5, 1, 2, 3, np.inf]
AGE_LABELS = ["0-6m", "6m-1y", "1-2y", "2-3y", ">3y"]
for label, group in DATASETS.items():
    metadata = pd.read_csv(DATA_DIR/group/"metadata"/(group+"_crossref_meta.csv"),
                           usecols=["id", "publication_date", "source"]).drop_duplicates("id", keep="first")
    metadata["date"] = pd.to_datetime(metadata.publication_date, format="%Y-%m-%d", errors="coerce")
    metadata["source_label"] = metadata.source.fillna("").astype(str).str.strip().str.lower()
    paper_data[label]["metadata"] = metadata


In [ ]:
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    adjusted[order] = np.minimum(1, np.maximum.accumulate(p_values[order]*(len(p_values)-np.arange(len(p_values)))))
    return adjusted

def prepare_age_sample(dataset):
    data = paper_data[dataset]
    frame = data["matched"].drop(columns="publication_date").merge(
        data["metadata"][["id", "date", "source_label"]], on="id", validate="one_to_one"
    )
    first = FIRST_COLLECTION[dataset]
    lower = first-pd.Timedelta(days=MAX_PUBLICATION_AGE_YEARS*AGE_DAYS_PER_YEAR)
    frame = frame[frame["date"].between(lower, first) & finite_counts(frame)].copy()
    frame = frame[(frame["total_citations_t1"] >= 0) & (frame["total_citations_t2"] >= frame["total_citations_t1"])].copy()
    frame["relative_growth"] = (frame["total_citations_t2"]-frame["total_citations_t1"])/frame["total_citations_t1"].clip(lower=1)
    frame["monthly_relative_growth"] = monthly_change(frame["relative_growth"], dataset)
    frame["age_years"] = (first-frame["date"]).dt.days/AGE_DAYS_PER_YEAR
    frame["age_group"] = pd.cut(frame["age_years"], AGE_EDGES, labels=AGE_LABELS)
    return frame[frame["age_group"].notna()].copy()

def age_comparison(samples):
    rows = []
    for age in AGE_LABELS:
        pred = samples["Predatory"].loc[samples["Predatory"]["age_group"] == age, "monthly_relative_growth"]
        norm = samples["Normal"].loc[samples["Normal"]["age_group"] == age, "monthly_relative_growth"]
        if not len(pred) or not len(norm):
            raise ValueError(f"Age group {age}: both datasets need eligible papers.")
        u, p = stats.mannwhitneyu(pred, norm, alternative="two-sided", method="asymptotic")
        rows.append({
            "Age group": age, "N Predatory": len(pred), "N Normal": len(norm),
            "Median Predatory (%/month)": 100*pred.median(), "Median Normal (%/month)": 100*norm.median(),
            "Median difference (pp/month)": 100*(pred.median()-norm.median()),
            "Mann–Whitney p": p, "Rank-biserial r": 2*u/(len(pred)*len(norm))-1,
        })
    result = pd.DataFrame(rows)
    result["Holm p"] = holm_adjust(result["Mann–Whitney p"])
    return result

age_samples = {label: prepare_age_sample(label) for label in DATASETS}
full = age_comparison(age_samples)
crossref = age_comparison({label: frame.loc[frame.source_label.eq("crossref")]
                          for label, frame in age_samples.items()})
table5 = full.merge(crossref, on="Age group", suffixes=(" full", " Crossref-only"), validate="one_to_one")
save_table(table5, "table5_age_growth")
